In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

RANDOM_STATE = 42


candidates = [Path("TSLA.csv"), Path("TSLA (1).csv")]
DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Place TSLA.csv in the same folder as this notebook "
        "(or rename your existing file to TSLA.csv)."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())


df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").drop_duplicates(subset="date").reset_index(drop=True)

quality_summary = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "data_type": df.dtypes.astype(str)
})

print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Duplicate dates:", df["date"].duplicated().sum())
display(quality_summary)


fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["date"], df["close"])
ax.set_title("Tesla Historical Closing Price")
ax.set_xlabel("Date")
ax.set_ylabel("Closing Price ($)")
ax.grid(alpha=0.25)
plt.show()


model_df = df.copy()

model_df["return_1d"] = model_df["close"].pct_change()
model_df["log_return_1d"] = np.log(model_df["close"] / model_df["close"].shift(1))
model_df["range_pct"] = (model_df["high"] - model_df["low"]) / model_df["close"]
model_df["open_close_pct"] = (model_df["close"] - model_df["open"]) / model_df["open"]
model_df["volume_change"] = model_df["volume"].pct_change()
model_df["volume_change"] = model_df["volume_change"].replace([np.inf, -np.inf], np.nan)

for lag in [1, 2, 3, 5, 10, 15]:
    model_df[f"return_lag_{lag}"] = model_df["return_1d"].shift(lag - 1)
    model_df[f"close_ratio_lag_{lag}"] = model_df["close"] / model_df["close"].shift(lag) - 1

for window in [3, 5, 10, 15, 20]:
    model_df[f"return_mean_{window}"] = model_df["return_1d"].rolling(window).mean()
    model_df[f"return_std_{window}"] = model_df["return_1d"].rolling(window).std()
    model_df[f"volume_ratio_{window}"] = (
        model_df["volume"] / model_df["volume"].rolling(window).mean()
    )
    model_df[f"close_ma_ratio_{window}"] = (
        model_df["close"] / model_df["close"].rolling(window).mean() - 1
    )

model_df["target_return"] = model_df["close"].shift(-1) / model_df["close"] - 1
model_df["target_close"] = model_df["close"].shift(-1)

feature_cols = [
    c for c in model_df.columns
    if (
        c in ["return_1d", "log_return_1d", "range_pct", "open_close_pct", "volume_change"]
        or c.startswith("return_lag_")
        or c.startswith("close_ratio_lag_")
        or c.startswith("return_mean_")
        or c.startswith("return_std_")
        or c.startswith("volume_ratio_")
        or c.startswith("close_ma_ratio_")
    )
]

model_df = model_df.dropna(
    subset=feature_cols + ["target_return", "target_close"]
).reset_index(drop=True)

print("Modelling rows:", len(model_df))
print("Number of features:", len(feature_cols))
print("Target: next-day percentage return")


split_idx = int(len(model_df) * 0.70)

train_df = model_df.iloc[:split_idx].copy()
test_df = model_df.iloc[split_idx:].copy()

X_train = train_df[feature_cols]
y_train = train_df["target_return"]

X_test = test_df[feature_cols]
y_test = test_df["target_return"]

print("Training period:", train_df["date"].min().date(), "to", train_df["date"].max().date())
print("Test period:", test_df["date"].min().date(), "to", test_df["date"].max().date())
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))


actual_close = test_df["target_close"].to_numpy()
naive_pred_close = test_df["close"].to_numpy()

naive_mae = mean_absolute_error(actual_close, naive_pred_close)
naive_rmse = np.sqrt(mean_squared_error(actual_close, naive_pred_close))

print(f"Naive baseline MAE:  ${naive_mae:,.2f}")
print(f"Naive baseline RMSE: ${naive_rmse:,.2f}")


param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [1, 2],
    "learning_rate": [0.01, 0.03, 0.05],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=1
    ),
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    n_jobs=1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print(f"Best CV return MAE: {-grid_search.best_score_:.5f}")


pred_return = best_model.predict(X_test)
pred_close = test_df["close"].to_numpy() * (1 + pred_return)

xgb_mae = mean_absolute_error(actual_close, pred_close)
xgb_rmse = np.sqrt(mean_squared_error(actual_close, pred_close))
xgb_r2 = r2_score(actual_close, pred_close)

direction_accuracy = np.mean(
    np.sign(pred_return) == np.sign(y_test.to_numpy())
)

print(f"XGBoost MAE:               ${xgb_mae:,.2f}")
print(f"XGBoost RMSE:              ${xgb_rmse:,.2f}")
print(f"XGBoost R²:                 {xgb_r2:.4f}")
print(f"Direction accuracy:         {direction_accuracy:.1%}")


comparison = pd.DataFrame({
    "Model": ["Naive: Previous Close", "XGBoost"],
    "MAE ($)": [naive_mae, xgb_mae],
    "RMSE ($)": [naive_rmse, xgb_rmse]
}).set_index("Model")

display(comparison.round(4))


results = pd.DataFrame({
    "date": test_df["date"],
    "actual_close": actual_close,
    "xgboost_predicted_close": pred_close,
    "naive_predicted_close": naive_pred_close
})

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(results["date"], results["actual_close"], label="Actual")
ax.plot(results["date"], results["xgboost_predicted_close"], label="XGBoost")
ax.plot(
    results["date"],
    results["naive_predicted_close"],
    label="Previous-close baseline",
    alpha=0.7
)

ax.set_title("Tesla Next-Day Closing Price: Actual vs Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Closing Price ($)")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


feature_importance = (
    pd.Series(best_model.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
    .head(12)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(9, 6))
feature_importance.plot(kind="barh", ax=ax)
ax.set_title("Top XGBoost Feature Importances")
ax.set_xlabel("Importance")
plt.show()

display(feature_importance.sort_values(ascending=False).to_frame("importance"))


results["absolute_error"] = (
    results["actual_close"] - results["xgboost_predicted_close"]
).abs()

display(results.tail(10).round(2))
